# Assignment 2 — Naive Bayes & KNN Experiments (Spambase)

Generic `assn2_experiment()` function — works on any dataset's preprocessed train/test split. Dataset used here: `Spambase_Dataset.csv`, target column `spam`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score
)

from sklearn.preprocessing import MinMaxScaler, Binarizer

from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

**Note:** `assn1_preprocess()` (Assignment 1's function) is included below because Assignment 2 needs its cleaned/scaled `X_train, X_test, y_train, y_test` as input — the actual Assignment-2-specific logic is entirely inside `assn2_experiment()` further down.

## Assignment 1 — `assn1_preprocess()`\n\nGeneric preprocessing function, broken into **7 numbered steps** so it's easy to walk through in an exam: inspect -> remove duplicates -> handle missing values -> split features/target -> EDA -> scale -> train/test split. Works on **any** dataset — just change `df` and `target_column`.

In [ ]:
def assn1_preprocess(
    df,
    target_column,
    scale=True,
    remove_duplicates=True,
    test_size=0.20,
    random_state=42
):
    """
    ==================================================================
    ASSIGNMENT 1 — GENERIC PREPROCESSING PIPELINE
    ==================================================================
    This ONE function is meant to work on ANY tabular classification
    dataset. You just pass in a dataframe and tell it which column is
    the target (the thing you're trying to predict), and it does all
    the standard "get the data ready for ML" work for you.

    Parameters
    ----------
    df : pandas DataFrame
        The raw dataset, exactly as read from a CSV.
    target_column : str
        Name of the column you want to predict (e.g. "spam").
    scale : bool
        Whether to squash all features into the 0-1 range using
        MinMaxScaler. Almost always True for KNN/NB style algorithms
        because they are sensitive to feature scale.
    remove_duplicates : bool
        Whether to drop exact duplicate rows before training.
    test_size : float
        Fraction of data held out for testing (0.20 = 20%).
    random_state : int
        A "seed" so that the random train/test split is reproducible
        (same split every time you run the notebook).

    Returns
    -------
    dict with keys: df, X, y, X_train, X_test, y_train, y_test, scaler
    """

    # ==============================================================
    # STEP 1: INSPECT THE DATA
    # ==============================================================
    # Before touching anything, always LOOK at the data first. This
    # tells us: how many rows/columns, what types of columns we have,
    # whether anything is obviously missing or broken.
    print("=" * 70)
    print("DATASET INFORMATION")
    print("=" * 70)

    # df.shape -> (number_of_rows, number_of_columns)
    print("\nShape :", df.shape)

    # df.head() shows the first 5 rows so we can eyeball the data
    display(df.head())

    print("\nColumn Names")
    print(df.columns.tolist())

    print()
    # df.info() shows column dtypes (int/float/object) and how many
    # non-null values each column has -> quick way to spot missing data
    df.info()

    print("\nStatistical Summary")
    # df.describe() gives mean/std/min/max/quartiles for numeric columns
    # (and count/unique/top/freq for categorical ones, via include="all")
    display(df.describe(include="all"))

    print("\nMissing Values Before Cleaning")
    # isnull() marks every cell True/False depending on whether it's
    # missing; .sum() adds those up per column -> count of NaNs per column
    print(df.isnull().sum())

    print("\nDuplicate Rows :", df.duplicated().sum())
    # duplicated() flags rows that are IDENTICAL to an earlier row

    # ==============================================================
    # STEP 2: REMOVE DUPLICATES
    # ==============================================================
    # Why remove duplicates? If the same row appears twice, the model
    # effectively "sees" that example twice, giving it more importance
    # than it deserves and slightly biasing training. Also duplicate
    # rows can leak from train into test if we're not careful, which
    # would make our test accuracy look better than it really is.
    if remove_duplicates:
        df = df.drop_duplicates().reset_index(drop=True)
        # reset_index(drop=True) re-numbers the rows 0,1,2,... after
        # dropping some — otherwise the index would have gaps in it.

    # ==============================================================
    # STEP 3: HANDLE MISSING VALUES
    # ==============================================================

    # ---- 3a. Drop rows where the TARGET itself is missing ----
    # If we don't know the true label (spam or not spam) for a row,
    # that row is USELESS for supervised learning — we can't train on
    # it and we can't fairly test on it. So we just remove those rows.
    if df[target_column].isnull().sum() > 0:
        print(f"\nRemoving {df[target_column].isnull().sum()} rows with missing target values")
        df = df.dropna(subset=[target_column]).reset_index(drop=True)

    # ---- 3b. Fill missing FEATURE values ----
    # First, separate columns into numeric vs categorical, because we
    # fill them differently.
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()

    # Numeric columns -> fill missing values with the MEDIAN.
    # Why median and not mean? The median is robust to outliers.
    # e.g. if most values are around 10 but one row is 10,000 (an
    # outlier/typo), the mean gets dragged way up, but the median
    # barely moves. So median is the safer "typical value" to plug in.
    for col in numeric_cols:
        if col != target_column:
            df[col] = df[col].fillna(df[col].median())

    # Categorical columns -> fill missing values with the MODE
    # (the most frequently occurring category). This is the natural
    # equivalent of "median" for non-numeric data — you can't average
    # text categories, but you can find the most common one.
    for col in categorical_cols:
        if col != target_column:
            df[col] = df[col].fillna(df[col].mode()[0])

    print("\nMissing Values After Cleaning")
    print(df.isnull().sum())  # should all be 0 now

    # ==============================================================
    # STEP 4: SPLIT FEATURES / TARGET
    # ==============================================================
    # X = everything the model is allowed to look at (the "inputs")
    # y = the thing we're trying to predict (the "answer")
    # This split is required before scaling and train/test splitting,
    # because we never want to accidentally scale or leak the target.
    X = df.drop(columns=[target_column])
    y = df[target_column]

    # ==============================================================
    # STEP 5: EDA (Exploratory Data Analysis) PLOTS
    # ==============================================================
    # EDA just means "look at the data visually before modelling" so
    # we understand what we're working with (balanced classes? skewed
    # features? correlated features?).

    # ---- Class distribution ----
    # A bar count of how many rows belong to each class. If one class
    # has 95% of the rows and the other has 5%, that's an IMBALANCED
    # dataset, and plain accuracy becomes a misleading metric (a lazy
    # model that always predicts the majority class would still score
    # 95% "accuracy" while being useless).
    plt.figure(figsize=(6, 5))
    sns.countplot(x=y)
    plt.title("Class Distribution")
    plt.show()

    # ---- Histograms ----
    # One histogram per numeric column, showing how its values are
    # spread out (normal-shaped? skewed? has outliers?).
    df.hist(figsize=(20, 18), bins=20)
    plt.tight_layout()
    plt.show()

    # ---- Boxplots ----
    # A boxplot shows the median (middle line), the interquartile
    # range (the box), and outliers (dots beyond the "whiskers").
    # Great for spotting outliers at a glance. We only plot the first
    # 10 numeric columns so the chart doesn't become unreadable when
    # a dataset has 50+ features.
    plt.figure(figsize=(18, 6))
    sns.boxplot(data=df.select_dtypes(include=np.number).iloc[:, :min(10, len(numeric_cols))])
    plt.xticks(rotation=90)
    plt.title("Boxplots")
    plt.show()

    # ---- Correlation heatmap ----
    # Correlation measures how strongly two numeric columns move
    # together, from -1 (perfectly opposite) to +1 (perfectly
    # together), with 0 meaning no linear relationship. This heatmap
    # helps us spot features that are strongly related to each other
    # (redundant info) or strongly related to the target (useful
    # predictors).
    plt.figure(figsize=(15, 12))
    sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm")
    plt.title("Correlation Heatmap")
    plt.show()

    # ==============================================================
    # STEP 6: SCALE FEATURES
    # ==============================================================
    # Why scale? Many ML algorithms (KNN, Naive Bayes with continuous
    # features, gradient-based models) work by measuring DISTANCES or
    # relying on the raw magnitude of numbers. If one feature ranges
    # 0-1 and another ranges 0-100000, the second feature will
    # completely dominate the distance calculation just because its
    # numbers are bigger — not because it's actually more important.
    # Scaling puts every feature on the same footing.
    #
    # MinMaxScaler specifically squashes every feature into [0, 1]
    # using the formula: (x - min) / (max - min)
    scaler = None
    if scale:
        scaler = MinMaxScaler()
        X = scaler.fit_transform(X)
        # fit_transform() does two things at once:
        #   fit  -> learns the min and max of each column from X
        #   transform -> actually applies the (x-min)/(max-min) formula
    else:
        X = X.values  # convert DataFrame to a plain numpy array

    # ==============================================================
    # STEP 7: TRAIN / TEST SPLIT
    # ==============================================================
    # We can't test a model on the same data it was trained on — the
    # model could just "memorize" the answers and look artificially
    # perfect. So we hold back a chunk of the data (test_size, e.g.
    # 20%) that the model NEVER sees during training, and only use it
    # at the very end to check how well the model generalizes to new,
    # unseen data.
    #
    # stratify=y is important for classification: it makes sure the
    # train set and test set both keep roughly the SAME proportion of
    # each class as the original data. Without it, a random split
    # could accidentally put almost all of one class into the test
    # set, giving a misleading evaluation.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    print("\nTraining Shape :", X_train.shape)
    print("Testing Shape  :", X_test.shape)
    print("\nTraining Class Distribution")
    print(y_train.value_counts())
    print("\nTesting Class Distribution")
    print(y_test.value_counts())

    # ==============================================================
    # RETURN everything downstream code will need, bundled in a dict
    # so we don't have to keep passing around 8 separate variables.
    # ==============================================================
    return {
        "df": df,
        "X": X,
        "y": y,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "scaler": scaler
    }

### Usage — Assignment 1

Swap `"spambase_updated.csv"` and `"spam"` for any other dataset/target and it works the same way.

In [ ]:
df = pd.read_csv("Spambase_Dataset.csv")

data = assn1_preprocess(df, "spam")

X_train = data["X_train"]
X_test  = data["X_test"]
y_train = data["y_train"]
y_test  = data["y_test"]
X = data["X"]
y = data["y"]
df = data["df"]

## Assignment 2 — `assn2_experiment()`\n\nGeneric experiment pipeline, broken into **7 numbered steps** so it's easy to walk through in an exam: Naive Bayes -> KNN across k -> Best KNN -> Grid/RandomizedSearch -> KDTree vs BallTree -> 5-fold CV -> final comparison table. Works on **any** dataset's `X_train, X_test, y_train, y_test` from `assn1_preprocess()`.\n\nThe `evaluate_model()` helper takes everything it needs as arguments (model, name, data, labels) — nothing hidden — so you can trace exactly what it does.

In [ ]:
def assn2_experiment(
    X_train, X_test, y_train, y_test,
    k_values=[1, 3, 5, 7, 9, 11],
    cv_folds=5,
    binarize_threshold=0.5,
    random_state=42
):
    """
    ==================================================================
    ASSIGNMENT 2 — NAIVE BAYES + KNN EXPERIMENT PIPELINE
    ==================================================================
    Takes the X_train, X_test, y_train, y_test produced by
    assn1_preprocess() and runs a full comparison of classification
    algorithms on them: 3 flavours of Naive Bayes, and KNN (tuned
    several different ways). Works on ANY dataset — nothing here is
    specific to Spambase.

    STEP 1: Naive Bayes (Gaussian, Multinomial, Bernoulli)
    STEP 2: KNN across multiple k values
    STEP 3: Best KNN
    STEP 4: GridSearchCV vs RandomizedSearchCV
    STEP 5: KDTree vs BallTree
    STEP 6: 5-fold Cross Validation (Naive Bayes + Best KNN)
    STEP 7: Final comparison table
    """

    results = []  # every model we evaluate adds one row (dict) here

    # ------------------------------------------------------------
    # evaluate_model() — a reusable helper so we don't repeat the
    # same 10 lines of "fit, time it, predict, time it, score it,
    # plot it" for every single model we try.
    #
    # It takes EVERYTHING it needs as arguments (model, name, the
    # data, the labels) instead of quietly reaching for variables
    # defined outside itself — that way, reading the function
    # signature alone tells you exactly what it depends on.
    # ------------------------------------------------------------
    def evaluate_model(model, name, Xtr, Xte, ytr, yte):

        # ---- TRAINING TIME ----
        # time.time() gives the current time in seconds. We record it
        # right before and right after model.fit(), and the difference
        # is how long training took.
        start = time.time()
        model.fit(Xtr, ytr)          # model "learns" from the training data
        train_time = time.time() - start

        # ---- PREDICTION TIME ----
        start = time.time()
        pred = model.predict(Xte)     # model guesses labels for unseen test data
        predict_time = time.time() - start

        # ---- PROBABILITY SCORES (needed for ROC-AUC) ----
        # predict_proba() returns, for each row, the probability of
        # belonging to each class. [:, 1] takes just the probability
        # of the POSITIVE class (e.g. "is spam"), which is what
        # roc_auc_score expects.
        prob = model.predict_proba(Xte)[:, 1]
        roc = roc_auc_score(yte, prob)

        # ---- METRICS EXPLAINED ----
        # Accuracy  = fraction of predictions that were correct overall.
        #             Misleading on imbalanced data (see assn1 EDA note).
        # Precision = of everything the model labelled "positive", how
        #             many actually WERE positive. High precision =
        #             few false alarms.
        # Recall    = of everything that actually WAS positive, how many
        #             did the model catch. High recall = few missed cases.
        # F1        = harmonic mean of precision & recall — a single
        #             number that balances both (useful when you care
        #             about both false alarms AND missed cases).
        # ROC-AUC   = how well the model ranks positive examples above
        #             negative ones, across ALL possible thresholds.
        #             1.0 = perfect ranking, 0.5 = random guessing.
        row = {
            "Model": name,
            "Accuracy": accuracy_score(yte, pred),
            "Precision": precision_score(yte, pred, zero_division=0),
            "Recall": recall_score(yte, pred, zero_division=0),
            "F1": f1_score(yte, pred, zero_division=0),
            "ROC-AUC": roc,
            "Train Time": train_time,
            "Predict Time": predict_time
        }
        results.append(row)

        # classification_report() prints precision/recall/F1 per class
        # in one readable block — handy for the report.
        print(f"\n{'='*60}\n{name}\n{'='*60}")
        print(classification_report(yte, pred, zero_division=0))

        # A confusion matrix shows, in a grid:
        #   rows    = the TRUE class
        #   columns = the class the model PREDICTED
        # The diagonal = correct predictions. Everything off the
        # diagonal = mistakes (and tells you WHICH kind of mistake).
        ConfusionMatrixDisplay.from_predictions(yte, pred)
        plt.title(f"Confusion Matrix - {name}")
        plt.show()

        return row

    # ==============================================================
    # STEP 1: NAIVE BAYES — Gaussian, Multinomial, Bernoulli
    # ==============================================================
    # Naive Bayes is a probability-based classifier built on Bayes'
    # Theorem. It's called "naive" because it assumes every feature
    # is independent of every other feature given the class — which
    # is almost never exactly true in real data, but the algorithm
    # still works surprisingly well in practice, and it's very fast.
    #
    # The three variants below differ in what KIND of feature data
    # they assume they're looking at:

    # -- GaussianNB --
    # Assumes each feature, within each class, follows a normal
    # (bell-curve) distribution. Good default choice for continuous
    # numeric features.
    evaluate_model(GaussianNB(), "Gaussian NB", X_train, X_test, y_train, y_test)

    # -- MultinomialNB --
    # Designed for COUNT-like data (e.g. word frequencies) and
    # requires all feature values to be non-negative — negative
    # numbers don't make sense as "counts". We check for that here so
    # the function doesn't crash on a dataset where scaling produced
    # negative values (e.g. if StandardScaler was used instead of
    # MinMaxScaler upstream).
    if X_train.min() >= 0 and X_test.min() >= 0:
        evaluate_model(MultinomialNB(), "Multinomial NB", X_train, X_test, y_train, y_test)
    else:
        print("Skipping Multinomial NB: negative feature values present.")

    # -- BernoulliNB --
    # Designed for BINARY (0/1, "present vs absent") features. Our
    # features are continuous (e.g. 0 to 1 after MinMax scaling), so
    # we first convert them to 0/1 using a Binarizer: any value above
    # `binarize_threshold` becomes 1 ("present"), anything at or below
    # becomes 0 ("absent"). This is a Bernoulli-specific step — the
    # other two Naive Bayes variants don't need it.
    binarizer = Binarizer(threshold=binarize_threshold)
    X_train_bin = binarizer.fit_transform(X_train)
    X_test_bin = binarizer.transform(X_test)
    evaluate_model(BernoulliNB(), "Bernoulli NB", X_train_bin, X_test_bin, y_train, y_test)

    # ==============================================================
    # STEP 2: KNN ACROSS MULTIPLE k VALUES
    # ==============================================================
    # KNN (K-Nearest Neighbors) classifies a new point by looking at
    # its "k" closest neighbours (by distance) in the training data,
    # and taking a majority vote of their labels.
    #
    # "k" is a hyperparameter WE choose, and it matters a lot:
    #   - too small k (e.g. k=1)  -> very sensitive to noise/outliers,
    #                                 can overfit
    #   - too large k             -> decision boundary becomes too
    #                                 smooth, can underfit
    # So instead of guessing one k, we try several and compare.
    knn_accuracies = []

    for k in k_values:
        model = KNeighborsClassifier(n_neighbors=k)
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        acc = accuracy_score(y_test, pred)
        knn_accuracies.append(acc)

    knn_table = pd.DataFrame({"k": k_values, "Accuracy": knn_accuracies})
    print("\nKNN Accuracy across k values")
    display(knn_table)

    # Plotting accuracy against k lets us visually see the "sweet
    # spot" — usually accuracy rises, plateaus, then may dip again as
    # k gets too large.
    plt.figure(figsize=(7, 5))
    plt.plot(k_values, knn_accuracies, "o-")
    plt.xlabel("k")
    plt.ylabel("Accuracy")
    plt.title("KNN Accuracy vs k")
    plt.grid(True)
    plt.show()

    # ==============================================================
    # STEP 3: BEST KNN (the k with the highest test accuracy)
    # ==============================================================
    # np.argmax() returns the INDEX of the largest value in a list —
    # here, the index of the highest accuracy — which we use to look
    # up the corresponding k.
    best_k = k_values[int(np.argmax(knn_accuracies))]
    print("\nBest k :", best_k)

    evaluate_model(
        KNeighborsClassifier(n_neighbors=best_k),
        f"Best KNN (k={best_k})",
        X_train, X_test, y_train, y_test
    )

    # ==============================================================
    # STEP 4: GRIDSEARCHCV vs RANDOMIZEDSEARCHCV
    # ==============================================================
    # Manually trying k values one at a time (Step 2) only tunes ONE
    # hyperparameter. In practice we often want to tune SEVERAL
    # hyperparameters together (here: n_neighbors AND weights), and
    # we want each combination judged fairly using cross-validation
    # rather than a single train/test split.
    #
    # "weights" controls how neighbours vote:
    #   "uniform"  -> every neighbour's vote counts equally
    #   "distance" -> closer neighbours get a bigger vote (makes sense
    #                 — a neighbour right next to you should matter
    #                 more than one far away)
    param_grid = {"n_neighbors": list(range(1, 20, 2)), "weights": ["uniform", "distance"]}

    # -- GridSearchCV --
    # Exhaustively tries EVERY combination of the given hyperparameter
    # values, using cv-fold cross-validation to score each combo, and
    # keeps the best one. Guaranteed to find the best combo within the
    # grid you gave it, but can be slow if the grid is large.
    start = time.time()
    grid = GridSearchCV(KNeighborsClassifier(), param_grid, cv=cv_folds)
    grid.fit(X_train, y_train)
    grid_time = time.time() - start

    # -- RandomizedSearchCV --
    # Instead of trying every combination, it randomly SAMPLES a fixed
    # number of combinations (n_iter=10 here) from the search space.
    # Much faster for large search spaces, and often finds a result
    # nearly as good as GridSearchCV.
    start = time.time()
    random_search = RandomizedSearchCV(
        KNeighborsClassifier(),
        {"n_neighbors": list(range(1, 20)), "weights": ["uniform", "distance"]},
        n_iter=10, cv=cv_folds, random_state=random_state
    )
    random_search.fit(X_train, y_train)
    random_time = time.time() - start

    tuning_table = pd.DataFrame([
        {"Method": "GridSearchCV", "Best Params": grid.best_params_,
         "CV Accuracy": grid.best_score_, "Time": grid_time},
        {"Method": "RandomizedSearchCV", "Best Params": random_search.best_params_,
         "CV Accuracy": random_search.best_score_, "Time": random_time}
    ])
    print("\nHyperparameter Tuning Comparison")
    display(tuning_table)

    # ==============================================================
    # STEP 5: KDTREE vs BALLTREE
    # ==============================================================
    # KNN needs to find the "nearest" points, which naively means
    # comparing a new point to EVERY training point (this is called
    # "brute force" search, and it's slow for large datasets).
    #
    # KDTree and BallTree are two different data structures that
    # organise the training points in a tree shape, so the search can
    # skip over large chunks of points that are obviously too far
    # away — making prediction much faster on large datasets.
    #   - KDTree  works by repeatedly splitting the data along axes
    #     (like x, then y, then x again...). Works well for a small-
    #     to-moderate NUMBER of features.
    #   - BallTree groups points into nested "balls" (hyperspheres)
    #     instead of axis-aligned splits. Tends to handle
    #     high-dimensional data (many features) better than KDTree.
    #
    # We compare both using our chosen best_k.
    tree_rows = []
    for algo in ["kd_tree", "ball_tree"]:
        model = KNeighborsClassifier(n_neighbors=best_k, algorithm=algo)
        row = evaluate_model(model, f"KNN ({algo})", X_train, X_test, y_train, y_test)
        tree_rows.append(row)

    tree_table = pd.DataFrame(tree_rows)

    # ==============================================================
    # STEP 6: 5-FOLD CROSS VALIDATION — Naive Bayes + Best KNN
    # ==============================================================
    # A single train/test split gives one accuracy number, which can
    # be a bit lucky or unlucky depending on which rows happened to
    # land in the test set. K-Fold Cross Validation is a more robust
    # way to evaluate a model:
    #   1. Split the TRAINING data into "cv_folds" equal chunks
    #      (folds) — here, 5 folds.
    #   2. Train on 4 folds, test/validate on the 1 remaining fold.
    #   3. Repeat 5 times, each time leaving out a DIFFERENT fold as
    #      the validation set.
    #   4. Average the 5 scores -> a much more reliable estimate of
    #      how the model will perform on new data, because every row
    #      got a turn being "held out" and tested on.
    cv_models = {
        "Gaussian NB": GaussianNB(),
        "Best KNN": KNeighborsClassifier(n_neighbors=best_k)
    }

    cv_rows = []
    for name, model in cv_models.items():
        # cross_val_score() automatically does all the splitting,
        # training, and scoring described above, and returns one
        # score per fold (an array of length cv_folds).
        scores = cross_val_score(model, X_train, y_train, cv=cv_folds)

        row = {"Model": name}
        for fold_number, score in enumerate(scores, start=1):
            row[f"Fold {fold_number}"] = score
        row["Average"] = scores.mean()

        cv_rows.append(row)

    cv_table = pd.DataFrame(cv_rows).set_index("Model")
    print(f"\n{cv_folds}-Fold Cross Validation")
    display(cv_table)

    # ==============================================================
    # STEP 7: FINAL COMPARISON TABLE
    # ==============================================================
    # Every evaluate_model() call above appended one row to `results`.
    # We now turn that into one tidy table, sorted best-to-worst by
    # accuracy, so you can see at a glance which model won overall.
    results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False).reset_index(drop=True)
    print("\nFinal Model Comparison")
    display(results_df)

    return {
        "results_df": results_df,
        "knn_table": knn_table,
        "best_k": best_k,
        "tuning_table": tuning_table,
        "grid_search": grid,
        "random_search": random_search,
        "tree_table": tree_table,
        "cv_table": cv_table
    }

### Usage — Assignment 2

Works with any `X_train, X_test, y_train, y_test` coming out of `assn1_preprocess()` — Spambase or otherwise.

In [ ]:
exp2_output = assn2_experiment(X_train, X_test, y_train, y_test)

results_df = exp2_output["results_df"]
results_df.to_csv("Experiment2_Results.csv", index=False)
results_df